# Comparative Experiments

See the repository README and data/README.md before running. Outputs were cleared for publication.

In [ ]:
from pathlib import Path
import os
ROOT = Path.cwd() if (Path.cwd() / "notebooks").is_dir() else Path.cwd().parent
DATA_DIR = Path(os.environ.get("PROJECT_DATA_DIR", str(ROOT / "data")))
ARTIFACT_DIR = ROOT / "artifacts"
ARTIFACT_DIR.mkdir(exist_ok=True)


## DATA LOADING

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.cluster import KMeans
from sklearn.ensemble import IsolationForest
from sklearn.metrics import silhouette_score

import umap
import hdbscan

customer_features = pd.read_csv(str(DATA_DIR / "customer_features.csv"))
customer_ids      = customer_features['client_id']
features          = customer_features.drop('client_id', axis=1)

scaler          = StandardScaler()
features_scaled = scaler.fit_transform(features)

print("Customer features :", customer_features.shape)
print("Features scaled   :", features_scaled.shape)

## COMPLETE PCA

#### SELECT FEATURES

In [ ]:
pca_temp = PCA()
pca_temp.fit(features_scaled)

plt.figure(figsize=(10, 5))
plt.plot(range(1, features_scaled.shape[1] + 1),
         pca_temp.explained_variance_ratio_.cumsum(),
         marker='o', color='steelblue')
plt.axhline(y=0.90, color='red',   linestyle='--', label='90% variance')
plt.axhline(y=0.95, color='green', linestyle='--', label='95% variance')
plt.xlabel('Number of Components')
plt.ylabel('Cumulative Explained Variance')
plt.title('PCA — Elbow Plot (13 features)')
plt.legend()
plt.grid(True)
plt.tight_layout()
plt.show()

cumsum = pca_temp.explained_variance_ratio_.cumsum()
n_components_90 = next(i+1 for i, v in enumerate(cumsum) if v >= 0.90)
n_components_98 = next(i+1 for i, v in enumerate(cumsum) if v >= 0.98)

print(f"90% variance at : {n_components_90} components")
print(f"98% variance at : {n_components_98} components")

pca        = PCA(n_components=n_components_98)
pca_result = pca.fit_transform(features_scaled)

print(f"\nSelected        : {n_components_98} components")
print(f"Variance kept   : {pca.explained_variance_ratio_.sum()*100:.2f}%")
print(f"PCA output shape: {pca_result.shape}")

# BASELINE MODEL KMEANS

In [ ]:
inertia = []
K_range = range(2, 14)

for k in K_range:
    km = KMeans(n_clusters=k, random_state=42, n_init=10)
    km.fit(pca_result)
    inertia.append(km.inertia_)


plt.figure(figsize=(10, 5))
plt.plot(K_range, inertia, marker='o', color='steelblue')
plt.xlabel('Number of Clusters (K)')
plt.ylabel('Inertia')
plt.title('K-Means — Elbow Method')
plt.grid(True)
plt.tight_layout()
plt.show()


sil_scores = []
for k in K_range:
    km = KMeans(n_clusters=k, random_state=42, n_init=10)
    labels = km.fit_predict(pca_result)
    sil_scores.append(silhouette_score(pca_result, labels))


plt.figure(figsize=(10, 5))
plt.plot(K_range, sil_scores, marker='o', color='coral')
plt.xlabel('Number of Clusters (K)')
plt.ylabel('Silhouette Score')
plt.title('K-Means — Silhouette Score per K')
plt.grid(True)
plt.tight_layout()
plt.show()


best_k         = K_range[sil_scores.index(max(sil_scores))]
kmeans         = KMeans(n_clusters=best_k, random_state=42, n_init=10)
kmeans_labels  = kmeans.fit_predict(pca_result)
kmeans_sil     = silhouette_score(pca_result, kmeans_labels)

print(f"Best K              : {best_k}")
print(f"K-Means Silhouette  : {kmeans_sil:.4f}")

# UMAP

In [ ]:
reducer     = umap.UMAP(n_components=4, random_state=42)
umap_result = reducer.fit_transform(pca_result)

umap_df              = pd.DataFrame(umap_result, columns=['UMAP1', 'UMAP2','UMAP3', 'UMAP4'])
umap_df['client_id'] = customer_ids.values

plt.figure(figsize=(10, 7))
plt.scatter(umap_result[:, 0], umap_result[:, 1],
            alpha=0.6, s=15, color='steelblue')
plt.title('UMAP — Customer Spending Behavior (2D)')
plt.xlabel('UMAP 1')
plt.ylabel('UMAP 2')
plt.tight_layout()
plt.show()

print("UMAP output shape:", umap_result.shape)

## COMPARISON

In [ ]:

clusterer     = hdbscan.HDBSCAN(min_cluster_size=30, min_samples=10)
cluster_labels = clusterer.fit_predict(umap_result)

# Silhouette score — noise points (-1) exclude karo
mask          = cluster_labels != -1
hdbscan_sil   = silhouette_score(umap_result[mask], cluster_labels[mask])

print(f"Unique clusters     : {len(set(cluster_labels[cluster_labels != -1]))}")
print(f"Noise points        : {(cluster_labels == -1).sum()}")
print(f"HDBSCAN Silhouette  : {hdbscan_sil:.4f}")
print(f"K-Means Silhouette  : {kmeans_sil:.4f}")
print(f"\nWinner              : {'HDBSCAN' if hdbscan_sil > kmeans_sil else 'K-Means'}")


# MODELS

In [ ]:
customer_features['cluster'] = cluster_labels
cluster_profile = customer_features.groupby('cluster').agg({
    'avg_transaction_amount' : 'mean',
    'total_spend'            : 'mean',
    'transaction_count'      : 'mean',
    'online_txn_ratio'       : 'mean',
    'night_txn_ratio'        : 'mean',
    'merchant_diversity'     : 'mean',
    'error_rate'             : 'mean',
    'spend_to_income_ratio'  : 'mean',
}).round(3)
print("\nCluster Profiles:")
print(cluster_profile)

In [ ]:
cluster_names = {
    -1 : 'Noise / Outliers',
     0 : 'Moderate Spenders',
     1 : 'Debt-Stressed Spenders',
     2 : 'High Value Customers',
     3 : 'Online Shoppers',
}
customer_features['cluster_name'] = customer_features['cluster'].map(cluster_names)
umap_df['cluster']      = cluster_labels
umap_df['cluster_name'] = customer_features['cluster_name'].values

# Colored UMAP plot
colors_map = {
    'Noise / Outliers'       : 'grey',
    'Moderate Spenders'      : 'orange',
    'Debt-Stressed Spenders' : 'green',
    'High Value Customers'   : 'brown',
    'Online Shoppers'        : 'blue',
}

plt.figure(figsize=(12, 8))
for name, color in colors_map.items():
    mask = umap_df['cluster_name'] == name
    plt.scatter(umap_df.loc[mask, 'UMAP3'],
                umap_df.loc[mask, 'UMAP1'],
                c=color, label=name, alpha=0.7, s=20)
plt.title('Customer Spending Personalities (HDBSCAN)')
plt.xlabel('UMAP 1')
plt.ylabel('UMAP 2')
plt.legend(loc='lower right', fontsize=9)
plt.tight_layout()
plt.show()

In [ ]:
iso_forest           = IsolationForest(n_estimators=200,contamination=0.5,random_state=42)
anomaly_scores       = iso_forest.fit_predict(features_scaled)
anomaly_score_values = iso_forest.decision_function(features_scaled)

customer_features['anomaly']       = anomaly_scores
customer_features['anomaly_score'] = anomaly_score_values

print("Anomaly distribution:")
print(pd.Series(anomaly_scores).value_counts())
print(f"\nFlagged as fraud risk: {(anomaly_scores == -1).sum()} customers")

# Fraud labels load karo
fraud_per_customer = pd.read_csv( str(DATA_DIR / "fraud_check.csv"))

fraud_per_customer['is_high_fraud'] = (fraud_per_customer['fraud_txn_count'] > 5).astype(int)

validation = customer_features[['client_id', 'anomaly']].merge(fraud_per_customer[['client_id', 'is_high_fraud']],on='client_id', how='left')


validation['is_high_fraud'] = validation['is_high_fraud'].fillna(0).astype(int)

caught           = validation[(validation['anomaly'] == -1) & (validation['is_high_fraud'] == 1)]

total_high_fraud = validation['is_high_fraud'].sum()

print("\nValidation Results:")
print(pd.crosstab(validation['anomaly'], validation['is_high_fraud'],rownames=['Predicted'], colnames=['Actual Fraud']))
print(f"\nDetection rate: {round(len(caught)/total_high_fraud*100, 1)}%")


umap_df['anomaly'] = anomaly_scores

plt.figure(figsize=(12, 8))
plt.scatter(umap_df.loc[umap_df['anomaly'] == 1,  'UMAP1'],
            umap_df.loc[umap_df['anomaly'] == 1,  'UMAP2'],
            c='steelblue', label='Normal', alpha=0.5, s=15)
plt.scatter(umap_df.loc[umap_df['anomaly'] == -1, 'UMAP1'],
            umap_df.loc[umap_df['anomaly'] == -1, 'UMAP2'],
            c='red', label='Fraud Risk', alpha=0.9, s=40, marker='X')
plt.title('UMAP — Normal vs Fraud Risk Customers')
plt.xlabel('UMAP 1')
plt.ylabel('UMAP 2')
plt.legend()
plt.tight_layout()
plt.show()

# HYPERPARAMETER TUNING

In [ ]:
min_cluster_sizes = [5,10, 15, 20, 25, 30]
min_samples_list  = [3, 5, 7,10,15]

results = []

for mcs in min_cluster_sizes:
    for ms in min_samples_list:
        cl      = hdbscan.HDBSCAN(min_cluster_size=mcs, min_samples=ms)
        labels  = cl.fit_predict(umap_result)
        n_clusters = len(set(labels[labels != -1]))
        n_noise    = (labels == -1).sum()
        
        if n_clusters >= 2:
            mask = labels != -1
            sil  = silhouette_score(umap_result[mask], labels[mask])
        else:
            sil  = -1
            
        results.append({
            'min_cluster_size' : mcs,
            'min_samples'      : ms,
            'n_clusters'       : n_clusters,
            'n_noise'          : n_noise,
            'silhouette_score' : round(sil, 4)
        })

results_df = pd.DataFrame(results).sort_values('silhouette_score', ascending=False)
print(results_df.to_string(index=False))

# Best params
best = results_df.iloc[0]
print(f"\nBest min_cluster_size : {int(best['min_cluster_size'])}")
print(f"Best min_samples      : {int(best['min_samples'])}")
print(f"Best silhouette score : {best['silhouette_score']}")
print(f"Clusters found        : {int(best['n_clusters'])}")

In [ ]:
feature_names = features.columns.tolist()
loadings = pd.DataFrame(pca.components_.T,index=feature_names,columns=[f'PC{i+1}' for i in range(pca.n_components_)])


plt.figure(figsize=(10, 6))
loadings['PC1'].abs().sort_values(ascending=True).plot(kind='barh', color='steelblue', edgecolor='white')

plt.title('Feature Importance — PC1 (Highest Variance Component)')
plt.xlabel('Absolute Loading Value')
plt.tight_layout()
plt.show()


plt.figure(figsize=(10, 6))
loadings['PC2'].abs().sort_values(ascending=True).plot(kind='barh', color='coral', edgecolor='white')

plt.title('Feature Importance — PC2')
plt.xlabel('Absolute Loading Value')
plt.tight_layout()
plt.show()

# Top 5 most important features overall
avg_importance = loadings.abs().mean(axis=1).sort_values(ascending=False)
print("Top features across all components:")
print(avg_importance)

# Important Thing 

In [ ]:
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
import pandas as pd
import numpy as np

# Standardize features
scaler = StandardScaler()
X_scaled = scaler.fit_transform(features)

# PCA
pca = PCA()
X_pca = pca.fit_transform(X_scaled)

# Explained variance
explained_variance = pca.explained_variance_ratio_
cumulative_variance = np.cumsum(explained_variance)

# Number of components needed for 90% variance
n_components_90 = np.argmax(cumulative_variance >= 0.90) + 1

print("Components needed for 90% variance:", n_components_90)

# PCA Loadings
loadings = pd.DataFrame(
    pca.components_.T,
    columns=[f"PC{i+1}" for i in range(len(features.columns))],
    index=features.columns
)

# Top features driving PC1
print("\nTop features in PC1:")
print(loadings["PC1"].abs().sort_values(ascending=False).head(10))

# Top features driving PC2
print("\nTop features in PC2:")
print(loadings["PC2"].abs().sort_values(ascending=False).head(10))

In [ ]:
import hdbscan
import pandas as pd

clusterer = hdbscan.HDBSCAN(
    min_cluster_size=15,
    min_samples=5
)

labels = clusterer.fit_predict(umap_result)

print("Unique Labels:", pd.Series(labels).unique())
print("\nCluster Counts:")
print(pd.Series(labels).value_counts().sort_index())

noise_pct = (labels == -1).mean() * 100
print(f"\nNoise Percentage: {noise_pct:.2f}%")

In [ ]:
features["cluster"] = labels

cluster_profile = features.groupby("cluster").mean().round(2)

cluster_profile.T

In [ ]:
print(features["spend_to_income_ratio"].describe())

print("\nTop 10 values:")
print(features["spend_to_income_ratio"].sort_values(ascending=False).head(10))

In [ ]:
import numpy as np

customer_features["spend_to_income_ratio_capped"] = np.clip(
    customer_features["spend_to_income_ratio"],
    customer_features["spend_to_income_ratio"].quantile(0.01),
    customer_features["spend_to_income_ratio"].quantile(0.99)
)

print(customer_features["spend_to_income_ratio_capped"].describe())

In [ ]:
customer_features.head()

In [ ]:
features = customer_features.drop(columns=['client_id' , 'spend_to_income_ratio'] )
features.head()


In [ ]:
# Replace old feature



from sklearn.preprocessing import StandardScaler
scaler = StandardScaler()
X_scaled = scaler.fit_transform(features)

# UMAP
import umap

reducer = umap.UMAP(n_components=4,n_neighbors=30,min_dist=0.1,random_state=42)

umap_result = reducer.fit_transform(X_scaled)

# HDBSCAN
import hdbscan
import pandas as pd

clusterer = hdbscan.HDBSCAN(min_cluster_size=15, min_samples=5)

labels = clusterer.fit_predict(umap_result)

print("Unique Labels:", pd.Series(labels).unique())
print("\nCluster Counts:")
print(pd.Series(labels).value_counts().sort_index())

noise_pct = (labels == -1).mean() * 100
print(f"\nNoise Percentage: {noise_pct:.2f}%")

In [ ]:
from sklearn.metrics import silhouette_score

mask = labels != -1

score = silhouette_score(
    umap_result[mask],
    labels[mask]
)

print("Silhouette Score:", score)

In [ ]:
from sklearn.metrics import silhouette_score

for n in [5,10,15,20,30,50]:
    
    clusterer = hdbscan.HDBSCAN(
        min_cluster_size=n,
        min_samples=n//2
    )

    labels = clusterer.fit_predict(umap_result)

    if len(set(labels)) > 2:
        mask = labels != -1
        
        score = silhouette_score(
            umap_result[mask],
            labels[mask]
        )

        print(
            f"min_cluster_size={n}",
            f"clusters={len(set(labels))-1}",
            f"score={score:.3f}"
        )

In [ ]:
clusterer = hdbscan.HDBSCAN(
    min_cluster_size=20,
    min_samples=10
)

labels = clusterer.fit_predict(umap_result)

import pandas as pd

print(pd.Series(labels).value_counts().sort_index())

noise_pct = (labels == -1).mean() * 100
print(f"\nNoise % = {noise_pct:.2f}")


cluster_profile = (
    customer_features
    .groupby("cluster")
    .mean()
    .round(2)
)

cluster_profile.T